### Milestone 3: Feature Engineering and Model Tuning

This notebook focuses on enhancing our sentiment classification pipeline by introducing new features that capture deeper sentiment from review text, including Word2Vec embeddings and lexicon-based sentiment scores. We then apply hyperparameter tuning using PySpark’s MLlib to improve model performance on top of these engineered features. Both Logistic Regression and Random Forest models are evaluated, and their performance is compared using accuracy, F1 score, and confusion matrices.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, count, when, isnan, isnull, mean, stddev, min, max, length, avg, expr, when, desc, concat, lit, coalesce
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, FloatType
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import Window
import numpy as np
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.feature import Word2Vec
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.sql import Row


In [2]:
spark = SparkSession.builder \
    .appName("Amazon Reviews Training") \
    .config("spark.executor.memory", "4g") \
    .config("spark.driver.memory", "6g") \
    .config("spark.executor.cores", "1") \
    .config("spark.default.parallelism", "16") \
    .config("spark.sql.shuffle.partitions", "16") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/06 22:52:34 INFO SparkEnv: Registering MapOutputTracker
25/04/06 22:52:34 INFO SparkEnv: Registering BlockManagerMaster
25/04/06 22:52:34 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/04/06 22:52:34 INFO SparkEnv: Registering OutputCommitCoordinator


In [3]:
amazon_dataset = spark.read.parquet("gs://final-project-bucket-amazon/processed/final_dataset.parquet")
print(f"Loaded {amazon_dataset.count()} reviews from parquet")

Loaded 18110850 reviews from parquet


In [4]:
print(f"Loaded {amazon_dataset.count()} rows")
amazon_dataset.printSchema()

Loaded 18110850 rows
root
 |-- helpful_ratio: double (nullable = true)
 |-- has_votes: integer (nullable = true)
 |-- vine_binary: integer (nullable = true)
 |-- verified_purchase_binary: integer (nullable = true)
 |-- product_category_vec: vector (nullable = true)
 |-- review_year: integer (nullable = true)
 |-- review_month: integer (nullable = true)
 |-- review_dayofweek: integer (nullable = true)
 |-- features: vector (nullable = true)
 |-- label: integer (nullable = true)



In [20]:
processed_df = spark.read.parquet("gs://final-project-bucket-amazon/processed/all_reviews_processed.parquet")

print(f"Loaded {processed_df.count()} reviews from all_reviews_processed.parquet")
processed_df.printSchema()

Loaded 18110850 reviews from all_reviews_processed.parquet
root
 |-- marketplace: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- helpful_votes: integer (nullable = true)
 |-- total_votes: integer (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: date (nullable = true)
 |-- helpful_ratio: double (nullable = true)
 |-- has_votes: integer (nullable = true)
 |-- review_length: integer (nullable = true)
 |-- combined_review_text: string (nullable = true)
 |-- combined_text_length: integer (nullable = true)
 |-- has_meaningful_content: integer (nu

In [6]:
# Very simple sentiment lexicon
positive_words = {
    "good", "great", "excellent", "amazing", "love", "perfect", "fantastic", "wonderful", "satisfied", "awesome",
    "outstanding", "superb", "brilliant", "enjoyed", "best", "recommend", "liked", "positive", "delightful", "pleased",
    "happy", "impressive", "exceptional", "super", "flawless", "ideal", "appreciate", "beautiful", "worth", "solid"
}

negative_words = {
    "bad", "terrible", "awful", "poor", "disappointed", "hate", "worst", "horrible", "broken", "unusable",
    "boring", "cheap", "lame", "ridiculous", "waste", "problem", "issue", "delay", "slow", "negative",
    "useless", "difficult", "disgusting", "return", "refund", "inconvenient", "annoying", "frustrating", "low"
}


# UDF to calculate net sentiment score
def compute_sentiment_score(word_list):
    if not word_list:
        return 0
    pos_count = sum(1 for w in word_list if w in positive_words)
    neg_count = sum(1 for w in word_list if w in negative_words)
    return pos_count - neg_count

sentiment_score_udf = udf(compute_sentiment_score, IntegerType())

# Add the sentiment_score column
processed_df = processed_df.withColumn(
    "sentiment_score", sentiment_score_udf(col("filtered_words"))
)

# Preview the new feature
processed_df.select("combined_review_text", "filtered_words", "sentiment_score", "label").show(5, truncate=False)


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [7]:
# Define a UDF to compute positive ratio
def compute_positive_ratio(words):
    if not words:
        return 0.0
    pos_count = sum(1 for w in words if w in positive_words)
    return float(pos_count) / len(words)

positive_ratio_udf = udf(compute_positive_ratio, FloatType())

# Apply to DataFrame
processed_df = processed_df.withColumn("positive_ratio", positive_ratio_udf(col("filtered_words")))



In [8]:
# UDF to compute negative ratio
def compute_negative_ratio(words):
    if not words:
        return 0.0
    neg_count = sum(1 for w in words if w in negative_words)
    return float(neg_count) / len(words)

negative_ratio_udf = udf(compute_negative_ratio, FloatType())

# Apply to DataFrame
processed_df = processed_df.withColumn("negative_ratio", negative_ratio_udf(col("filtered_words")))


In [9]:
# Sample 10% for training
w2v_sample_df = processed_df.sample(False, 0.02, seed=42)

# Use vectorSize = 50 for speed/memory optimization
w2v = Word2Vec(vectorSize=50, minCount=5, inputCol="filtered_words", outputCol="w2v_vector")
w2v_model = w2v.fit(w2v_sample_df)

# Transform full dataset with learned model
processed_df = w2v_model.transform(processed_df)


In [10]:
final_feature_cols = [
    "features",               # TF-IDF
    "product_category_vec",   # One-hot category
    "helpful_ratio",
    "has_votes",
    "vine_binary",
    "verified_purchase_binary",
    "review_year",
    "review_month",
    "review_dayofweek",
    "sentiment_score",
    "positive_ratio",
    "negative_ratio",
    "w2v_vector"
]



assembler = VectorAssembler(
    inputCols=final_feature_cols,
    outputCol="final_features"
)

# This will give you a DataFrame with all features + all original columns
processed_df = assembler.transform(processed_df)

final_df = processed_df.select("final_features", "label")
# Just check schema to confirm the final structure
final_df.printSchema()

# Look at a few rows (without truncating the full vector)
final_df.select("final_features", "label").show(3, truncate=False)
# Save as Parquet (efficient for vector data)
final_df.write.mode("overwrite").parquet("gs://final-project-bucket-amazon/processed/final_vectorized_df.parquet")


root
 |-- final_features: vector (nullable = true)
 |-- label: integer (nullable = true)



25/04/06 23:01:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

25/04/06 23:01:38 WARN DAGScheduler: Broadcasting large task binary with size 1228.7 KiB


### Logistic Regression

In [12]:
# Split 85% for training + validation, 15% for final test
train_val_df, test_df = final_df.randomSplit([0.85, 0.15], seed=42)

# Split train_val into train (70%) and validation (15%)
train_df, val_df = train_val_df.randomSplit([0.8235, 0.1765], seed=42)

# Show counts
print(f"Train count: {train_df.count()}")
print(f"Validation count: {val_df.count()}")
print(f"Test count: {test_df.count()}")



Train count: 12674701


Validation count: 2717952


Test count: 2718197


In [13]:
# Count class instances
label_counts = train_df.groupBy("label").count().toPandas()

# Get label counts
neg_count = label_counts[label_counts['label'] == 0]['count'].values[0]
pos_count = label_counts[label_counts['label'] == 1]['count'].values[0]
total = neg_count + pos_count

# Inverse frequency
neg_weight = total / (2 * neg_count)
pos_weight = total / (2 * pos_count)

print(f"Class 0 (negative) weight: {neg_weight}")
print(f"Class 1 (positive) weight: {pos_weight}")

Class 0 (negative) weight: 2.149654961639414
Class 1 (positive) weight: 0.6515468421054255


In [14]:
train_df = train_df.withColumn("class_weight", when(col("label") == 0, lit(neg_weight)).otherwise(lit(pos_weight)))
val_df = val_df.withColumn("class_weight", when(col("label") == 0, lit(neg_weight)).otherwise(lit(pos_weight)))
test_df = test_df.withColumn("class_weight", when(col("label") == 0, lit(neg_weight)).otherwise(lit(pos_weight)))


In [15]:
# Define LR model with weightCol
lr = LogisticRegression(
    featuresCol="final_features",
    labelCol="label",
    predictionCol="prediction",
    weightCol="class_weight",
    maxIter=20
)

# Small grid for fast tuning
param_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01, 0.1, 1.0]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# Evaluator using F1 score
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

# Train-validation split
tvs = TrainValidationSplit(
    estimator=lr,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    seed=42
)

# Train on full train_df
tvs_model = tvs.fit(train_df)


25/04/03 06:36:15 WARN BlockManager: Asked to remove block broadcast_514_piece0, which does not exist


In [16]:
val_preds = tvs_model.transform(val_df)
test_preds = tvs_model.transform(test_df)

def evaluate(df, name):
    acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(df)
    f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(df)
    print(f"{name} — Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")

evaluate(val_preds, "Validation Set")
evaluate(test_preds, "Test Set")


Validation Set — Accuracy: 0.8684, F1 Score: 0.8741


Test Set — Accuracy: 0.8684, F1 Score: 0.8742


In [17]:
# Save the best model from TrainValidationSplit
tvs_model.bestModel.write().overwrite().save("gs://final-project-bucket-amazon/models/logistic_model_weighted_final")

In [18]:
test_preds.write.mode("overwrite").parquet("gs://final-project-bucket-amazon/results/logistic_preds_weighted.parquet")

25/04/03 07:27:24 WARN DAGScheduler: Broadcasting large task binary with size 1358.7 KiB


In [4]:
lr_model = LogisticRegressionModel.load("gs://final-project-bucket-amazon/models/logistic_model_weighted_final")

# Step 1: Get coefficients (1D vector of weights)
coefficients = lr_model.coefficients.toArray()

# Step 2: Get top N most important features (by absolute value)
importance_df = pd.DataFrame({
    "index": list(range(len(coefficients))),
    "coefficient": coefficients,
    "importance": abs(coefficients)
})

# Step 3: Show top 25 most important features
top_features = importance_df.sort_values("importance", ascending=False).head(25)
top_features


,index,coefficient,importance
10020,10020,-3.678188,3.678188
10014,10014,-3.437764,3.437764
10023,10023,-3.306025,3.306025
10044,10044,-3.073270,3.073270
10022,10022,-2.927749,2.927749
10050,10050,-2.747486,2.747486
10056,10056,-2.648972,2.648972
10040,10040,-2.592555,2.592555
10034,10034,2.339460,2.339460
10046,10046,-1.999165,1.999165


In [5]:
final_feature_cols = [
    "features",                 # HashingTF (10,000)
    "product_category_vec",     # one-hot category (let's say ~100)
    "helpful_ratio",            # scalar
    "has_votes",                # scalar
    "vine_binary",              # scalar
    "verified_purchase_binary", # scalar
    "review_year",              # scalar
    "review_month",             # scalar
    "review_dayofweek",         # scalar
    "sentiment_score",          # scalar
    "positive_ratio",           # scalar
    "w2v_vector"                # 50-dimensional dense vector
]


In [9]:
# Define sizes of each block
sizes = {
    "features": 10000,
    "product_category_vec": processed_df.select("product_category_vec").first()[0].size,
    "w2v_vector": 50
}

# Generate flat feature names
feature_names = []

# Add TF (hashed word features)
feature_names += [f"tf_{i}" for i in range(sizes["features"])]

# Add product category one-hot
feature_names += [f"category_{i}" for i in range(sizes["product_category_vec"])]

# Add scalar features (same order as in assembler)
feature_names += [
    "helpful_ratio",
    "has_votes",
    "vine_binary",
    "verified_purchase_binary",
    "review_year",
    "review_month",
    "review_dayofweek",
    "sentiment_score",
    "positive_ratio"
]

# Add w2v vector dims
feature_names += [f"w2v_{i}" for i in range(sizes["w2v_vector"])]


In [10]:
coeffs = lr_model.coefficients.toArray()

importance_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coeffs,
    "importance": abs(coeffs)
})

top_features = importance_df.sort_values("importance", ascending=False).head(25)
top_features


,feature,coefficient,importance
10020,w2v_6,-3.678188,3.678188
10014,w2v_0,-3.437764,3.437764
10023,w2v_9,-3.306025,3.306025
10044,w2v_30,-3.073270,3.073270
10022,w2v_8,-2.927749,2.927749
10050,w2v_36,-2.747486,2.747486
10056,w2v_42,-2.648972,2.648972
10040,w2v_26,-2.592555,2.592555
10034,w2v_20,2.339460,2.339460
10046,w2v_32,-1.999165,1.999165


In [3]:
# Load vectorized dataset from GCS
final_df = spark.read.parquet("gs://final-project-bucket-amazon/processed/final_vectorized_df.parquet")

# Check schema
final_df.printSchema()

# Quick peek at rows
final_df.show(5, truncate=False)


root
 |-- final_features: vector (nullable = true)
 |-- label: integer (nullable = true)



+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Random Forest 

In [4]:
# Split into train+val and test
train_val_df, test_df = final_df.randomSplit([0.85, 0.15], seed=42)

# Split train_val into train and validation
train_df, val_df = train_val_df.randomSplit([0.8235, 0.1765], seed=42)  # ~70/15/15 split

print(f"Train count: {train_df.count()}")
print(f"Validation count: {val_df.count()}")
print(f"Test count: {test_df.count()}")


Train count: 12674472


Validation count: 2717994


Test count: 2718384


In [ ]:
# Count each class
label_counts = train_df.groupBy("label").count().toPandas()
neg_count = label_counts[label_counts['label'] == 0]['count'].values[0]
pos_count = label_counts[label_counts['label'] == 1]['count'].values[0]

# Calculate downsample ratio
downsample_ratio = neg_count / pos_count

# Downsample
neg_class_df = train_df.filter(col("label") == 0)
pos_class_df = train_df.filter(col("label") == 1)
pos_class_downsampled = pos_class_df.sample(withReplacement=False, fraction=downsample_ratio, seed=42)

# Create balanced training set
train_balanced_df = neg_class_df.union(pos_class_downsampled)

print(f"Balanced training data count: {train_balanced_df.count()}")


Balanced training data count: 5895167


In [6]:
# Define Random Forest
rf = RandomForestClassifier(
    featuresCol="final_features",
    labelCol="label",
    predictionCol="prediction",
    seed=42
)

# Define parameter grid
param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [30, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

# Evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="f1"
)

# TrainValidationSplit
tvs = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    trainRatio=0.8,
    seed=42
)

# Train on balanced data
rf_model = tvs.fit(train_balanced_df)


25/04/05 00:28:56 WARN DAGScheduler: Broadcasting large task binary with size 1050.1 KiB
25/04/05 00:38:29 WARN DAGScheduler: Broadcasting large task binary with size 1071.3 KiB
25/04/05 00:40:02 WARN DAGScheduler: Broadcasting large task binary with size 1111.2 KiB
25/04/05 00:41:37 WARN DAGScheduler: Broadcasting large task binary with size 1191.1 KiB
25/04/05 00:43:15 WARN DAGScheduler: Broadcasting large task binary with size 1342.4 KiB
25/04/05 00:49:06 WARN DAGScheduler: Broadcasting large task binary with size 1050.1 KiB
25/04/05 00:58:32 WARN DAGScheduler: Broadcasting large task binary with size 1071.3 KiB
25/04/05 01:00:16 WARN DAGScheduler: Broadcasting large task binary with size 1111.2 KiB
25/04/05 01:02:01 WARN DAGScheduler: Broadcasting large task binary with size 1191.1 KiB
25/04/05 01:03:49 WARN DAGScheduler: Broadcasting large task binary with size 1342.4 KiB
25/04/05 01:05:41 WARN DAGScheduler: Broadcasting large task binary with size 1615.0 KiB
25/04/05 01:07:39 WAR

In [ ]:
# Predictions
train_preds = rf_model.transform(train_df)
val_preds = rf_model.transform(val_df)
test_preds = rf_model.transform(test_df)

# Evaluation function
def evaluate(df, name):
    acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(df)
    f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1").evaluate(df)
    print(f"{name} — Accuracy: {acc:.4f}, F1 Score: {f1:.4f}")

# Run evals
evaluate(train_preds, "Train Set")
evaluate(val_preds, "Validation Set")
evaluate(test_preds, "Test Set")


25/04/05 03:06:18 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
25/04/05 03:10:41 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB


Train Set — Accuracy: 0.7665, F1 Score: 0.7835


25/04/05 03:15:09 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
25/04/05 03:17:29 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB


Validation Set — Accuracy: 0.7662, F1 Score: 0.7831


25/04/05 03:19:44 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
25/04/05 03:21:53 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB


Test Set — Accuracy: 0.7668, F1 Score: 0.7837


In [8]:
# Save best model from TrainValidationSplit
rf_model.bestModel.write().overwrite().save("gs://final-project-bucket-amazon/models/rf_model_balanced")


In [11]:
logistic_test_preds = spark.read.parquet("gs://final-project-bucket-amazon/results/logistic_preds_weighted.parquet")


In [12]:
# Group and count label-prediction pairs
conf_matrix_df = logistic_test_preds.groupBy("label", "prediction").count().orderBy("label", "prediction")

# Show the result
conf_matrix_df.show()


+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0| 555643|
|    0|       1.0|  76720|
|    1|       0.0| 280936|
|    1|       1.0|1804898|
+-----+----------+-------+



In [13]:
conf_pivot_logistic = conf_matrix_df.groupBy("label") \
    .pivot("prediction", [0.0, 1.0]) \
    .sum("count") \
    .fillna(0) \
    .orderBy("label")

conf_pivot_logistic.show()


+-----+------+-------+
|label|   0.0|    1.0|
+-----+------+-------+
|    0|555643|  76720|
|    1|280936|1804898|
+-----+------+-------+



In [14]:
# Load the final trained RF model
rf_model = RandomForestClassificationModel.load("gs://final-project-bucket-amazon/models/rf_model_balanced")


In [15]:
# Load final vectorized dataset
final_df = spark.read.parquet("gs://final-project-bucket-amazon/processed/final_vectorized_df.parquet")

# Recreate split for test set
train_val_df, test_df = final_df.randomSplit([0.85, 0.15], seed=42)


In [16]:
test_preds = rf_model.transform(test_df)


In [17]:
# Group by label and prediction
conf_matrix = test_preds.groupBy("label", "prediction").count().orderBy("label", "prediction")
conf_matrix.show()

# Pivoted version for easier view
conf_matrix.groupBy("label").pivot("prediction", [0.0, 1.0]) \
    .sum("count").fillna(0).orderBy("label").show()


25/04/06 23:23:01 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
25/04/06 23:25:23 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB


+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|    0|       0.0| 518550|
|    0|       1.0| 114379|
|    1|       0.0| 725032|
|    1|       1.0|1360433|
+-----+----------+-------+



25/04/06 23:25:24 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
25/04/06 23:27:34 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB
25/04/06 23:27:35 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB


+-----+------+-------+
|label|   0.0|    1.0|
+-----+------+-------+
|    0|518550| 114379|
|    1|725032|1360433|
+-----+------+-------+



In [21]:
# This will give you the actual size of the one-hot encoded category vector
category_size = processed_df.select("product_category_vec").first()[0].size


In [22]:
# Start from the known order of your VectorAssembler
feature_names = [f"tf_{i}" for i in range(10000)]
feature_names += [f"category_{i}" for i in range(category_size)]
feature_names += [
    "helpful_ratio", "has_votes", "vine_binary", "verified_purchase_binary",
    "review_year", "review_month", "review_dayofweek",
    "sentiment_score", "positive_ratio", "negative_ratio"
]
feature_names += [f"w2v_{i}" for i in range(50)]


In [23]:
# Extract importancesfor random forest 
importances = rf_model.featureImportances.toArray()

# Trim feature_names if needed
if len(importances) < len(feature_names):
    feature_names = feature_names[:len(importances)]

# Convert to Spark DataFrame
importance_rows = [Row(feature=feature_names[i], importance=float(imp)) for i, imp in enumerate(importances)]
importance_df = spark.createDataFrame(importance_rows)

# Show top 25 features
importance_df.orderBy(col("importance").desc()).show(25, truncate=False)


+---------------+--------------------+
|feature        |importance          |
+---------------+--------------------+
|sentiment_score|0.05367280985327092 |
|w2v_29         |0.05176023077309377 |
|positive_ratio |0.03675677258412499 |
|negative_ratio |0.03600179805354796 |
|w2v_35         |0.034687276936365065|
|w2v_7          |0.03285298517130197 |
|w2v_9          |0.032250485917935466|
|w2v_42         |0.029854619477310572|
|w2v_30         |0.026276290201012558|
|w2v_5          |0.025808907219520887|
|w2v_24         |0.022997596170141195|
|tf_750         |0.02209018253299534 |
|w2v_39         |0.020374848886587797|
|w2v_47         |0.01836561000452036 |
|w2v_40         |0.017727467278299422|
|w2v_6          |0.016774450965335662|
|w2v_25         |0.0161242728245598  |
|w2v_17         |0.015563885567279117|
|w2v_14         |0.014030924574208332|
|w2v_16         |0.013840398378365705|
|tf_4012        |0.01375076848305279 |
|w2v_33         |0.01300283430778099 |
|w2v_38         |0.012794